# K-Fold Training v2 (No AMP)

This notebook runs no-AMP K-fold training and logs training-loss snapshots for visualization.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "twitter_sentiment_v2_noamp_debug"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("Git branch:", current_branch)
print("Git commit:", current_commit)
print("Repo ready at:", REPO_DIR)


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import (
    TweetExtractionCausalDataset,
    causal_collate_fn,
    register_special_tokens,
)
from dz2_causal.losses import compute_total_loss
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard

CFG_PATH = os.environ.get("CFG_PATH", "config/kaggle_train_kfold_small_16gb.json")
print("Using config:", CFG_PATH)
CFG = json.loads(Path(CFG_PATH).read_text())
CFG

assert bool(CFG.get("no_amp", False)) is True, "This notebook is no-AMP only. Set no_amp=true in config."
print("no_amp:", CFG["no_amp"], "| model:", CFG["model_name"])



In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

df = pd.read_csv(CFG["train_csv"]).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
print("Train rows:", len(df))

splits = list(
    StratifiedKFold(
        n_splits=CFG["n_splits"],
        shuffle=True,
        random_state=CFG["seed"],
    ).split(df, df["sentiment"])
)

OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def make_records(frame: pd.DataFrame):
    records = frame[["text", "sentiment", "selected_text"]].to_dict("records")
    for r in records:
        r["prompt"] = CFG["prompt_text"]
    return records


def align_head_dtypes_with_lm(model, device):
    target_dtype = model.lm.get_input_embeddings().weight.dtype
    model.start_head.to(device=device, dtype=target_dtype)
    model.end_head.to(device=device, dtype=target_dtype)
    model.select_head.to(device=device, dtype=target_dtype)
    return target_dtype


def run_smoke_step(model, loader, optimizer, scheduler, device):
    smoke_batch = next(iter(loader))
    for k, v in smoke_batch.items():
        if torch.is_tensor(v):
            smoke_batch[k] = v.to(device)

    out = model(
        input_ids=smoke_batch["input_ids"],
        attention_mask=smoke_batch["attention_mask"],
        labels=smoke_batch["labels"],
    )
    losses = compute_total_loss(
        outputs=out,
        batch=smoke_batch,
        lambda_kl=0.0,
        lambda_select=0.0,
    )
    loss = losses["loss"]
    print(f"[smoke] loss={loss.item():.4f} ce={losses['ce_loss'].item():.4f}")

    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()


def train_one_fold(fold_id: int, train_idx, val_idx):
    print(f"\n===== Fold {fold_id} =====")

    tokenizer = AutoTokenizer.from_pretrained(
        CFG["model_name"],
        use_fast=True,
        trust_remote_code=CFG.get("trust_remote_code", False),
    )

    model = CausalExtractionModel(
        model_name=CFG["model_name"],
        trust_remote_code=CFG.get("trust_remote_code", False),
    )
    register_special_tokens(tokenizer, model=model.lm)

    train_ds = TweetExtractionCausalDataset(
        records=make_records(df.iloc[train_idx].reset_index(drop=True)),
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        max_len=CFG["max_len"],
        soft_alpha=CFG["soft_alpha"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        shuffle=True,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: causal_collate_fn(b, tokenizer.pad_token_id),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    lm_dtype = align_head_dtypes_with_lm(model, device)
    print(f"Aligned heads to LM dtype: {lm_dtype}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
    )

    total_steps = max(1, CFG["epochs"] * len(train_loader))
    warmup_steps = int(CFG["warmup_ratio"] * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    optimizer.zero_grad(set_to_none=True)

    ckpt_root = OUTPUT_DIR / "checkpoints" / f"fold{fold_id}"
    ckpt_root.mkdir(parents=True, exist_ok=True)

    def save_ckpt(tag: str, epoch_id: int, step_id: int, gstep: int):
        ckpt = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_name": CFG["model_name"],
            "config": CFG,
            "fold": int(fold_id),
            "epoch": int(epoch_id),
            "step": int(step_id),
            "global_step": int(gstep),
        }
        ckpt_path = ckpt_root / f"{tag}.pt"
        torch.save(ckpt, ckpt_path)
        latest_path = ckpt_root / "latest.pt"
        torch.save(ckpt, latest_path)
        return ckpt_path

    if CFG.get("run_smoke_test", True):
        run_smoke_step(model, train_loader, optimizer, scheduler, device)

    history = []
    global_step = 0

    model.train()
    for epoch in range(CFG["epochs"]):
        ce_only = epoch < CFG["ce_only_epochs"]
        lambda_kl = 0.0 if ce_only else CFG["lambda_kl"]
        lambda_select = 0.0 if ce_only else CFG["lambda_select"]

        running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}
        for step, batch in enumerate(train_loader):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)

            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            losses = compute_total_loss(
                out,
                batch,
                lambda_kl=lambda_kl,
                lambda_select=lambda_select,
            )
            loss = losses["loss"] / CFG["grad_accum_steps"]
            loss.backward()

            if (step + 1) % CFG["grad_accum_steps"] == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1


            running["loss"] += losses["loss"].item()
            running["ce"] += losses["ce_loss"].item()
            running["kl"] += losses["kl_loss"].item()
            running["sel"] += losses["select_loss"].item()

            if (step + 1) % CFG["log_every"] == 0:
                denom = float(CFG["log_every"])
                loss_row = {
                    "fold": int(fold_id),
                    "epoch": int(epoch + 1),
                    "step": int(step + 1),
                    "global_step": int(global_step),
                    "loss": float(running["loss"] / denom),
                    "ce": float(running["ce"] / denom),
                    "kl": float(running["kl"] / denom),
                    "sel": float(running["sel"] / denom),
                }
                history.append(loss_row)

                print(
                    f"fold={fold_id} epoch={epoch+1} step={step+1}/{len(train_loader)} "
                    f"loss={loss_row['loss']:.4f} ce={loss_row['ce']:.4f} "
                    f"kl={loss_row['kl']:.4f} sel={loss_row['sel']:.4f}"
                )
                running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}

        epoch_ckpt_path = save_ckpt(
            tag=f"epoch_{epoch + 1}",
            epoch_id=epoch + 1,
            step_id=len(train_loader),
            gstep=global_step,
        )
        print(f"Saved epoch checkpoint: {epoch_ckpt_path}")

    val_df = df.iloc[val_idx].reset_index(drop=True)
    eval_out = evaluate_dataframe_jaccard(
        df=val_df,
        model=model,
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        device=device,
        max_new_tokens=CFG["max_new_tokens"],
    )

    ckpt_path = OUTPUT_DIR / f"model_fold{fold_id}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_name": CFG["model_name"],
            "fold": fold_id,
            "val_jaccard": eval_out["mean_jaccard"],
            "config": CFG,
            "global_step": int(global_step),
        },
        ckpt_path,
    )

    print(f"Fold {fold_id} val_jaccard={eval_out['mean_jaccard']:.4f} | saved: {ckpt_path}")
    return {
        "fold": fold_id,
        "val_jaccard": eval_out["mean_jaccard"],
        "checkpoint": str(ckpt_path),
        "checkpoint_dir": str(ckpt_root),
        "history": history,
    }


In [ ]:
fold_results = []
for fold_id, (train_idx, val_idx) in enumerate(splits):
    result = train_one_fold(fold_id, train_idx, val_idx)
    fold_results.append(result)

    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_summary = [
    {
        "fold": r["fold"],
        "val_jaccard": r["val_jaccard"],
        "checkpoint": r["checkpoint"],
    }
    for r in fold_results
]
metrics_path = OUTPUT_DIR / "kfold_metrics.json"
metrics_path.write_text(json.dumps(metrics_summary, indent=2))
print("Saved metrics:", metrics_path)

history_rows = []
for r in fold_results:
    history_rows.extend(r.get("history", []))

history_df = pd.DataFrame(history_rows)
history_path = OUTPUT_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved training history:", history_path)

fold_results


In [ ]:
scores = [r["val_jaccard"] for r in fold_results]
print("Mean Jaccard:", float(np.mean(scores)))
print("Std Jaccard:", float(np.std(scores)))

if len(history_df) == 0:
    print("No training history points were logged. Reduce CFG['log_every'] to capture curves.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    metric_specs = [
        ("loss", "Total Loss"),
        ("ce", "CE Loss"),
        ("kl", "KL Loss"),
        ("sel", "Select Loss"),
    ]

    for ax, (col, title) in zip(axes.flat, metric_specs):
        for fold_id, fold_hist in history_df.groupby("fold"):
            ax.plot(fold_hist["global_step"], fold_hist[col], label=f"fold {int(fold_id)}")
        ax.set_title(title)
        ax.set_xlabel("Global Step")
        ax.set_ylabel(col)
        ax.grid(alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(5, len(labels)))
    fig.suptitle("Training Loss Curves by Fold", y=1.02)
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
fold_ids = [int(r["fold"]) for r in fold_results]
fold_scores = [float(r["val_jaccard"]) for r in fold_results]
ax.bar(fold_ids, fold_scores)
ax.set_title("Validation Jaccard by Fold")
ax.set_xlabel("Fold")
ax.set_ylabel("Jaccard")
ax.set_ylim(0.0, max(1.0, max(fold_scores) + 0.05))
ax.grid(axis="y", alpha=0.25)
plt.show()
